In [81]:
!pip install PyMuPDF

import fitz

In [82]:
# Extracts PDF text with pages
def read_pdf_with_page_numbers(file_path):
  doc = fitz.open(file_path)
  data = []
  for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text")
    data.append({"page": page_num+1, "content": text})

  return data;

pages = read_pdf_with_page_numbers("/content/drive/MyDrive/JOB TASK DATASET/policy_file.pdf");
print(pages[0])

{'page': 1, 'content': '1.2 \nFINANCIAL POLICY OBJECTIVES AND STRATEGIES \n \nSTATEMENT \nThe presentation and preparation of the Territory’s Budget is provided for in sections 11 and \n11A of the Financial Management Act 1996 (the Act).   \nThe purpose of the financial policy objectives and strategies statement is to make transparent \nthe Government’s financial strategies and to establish a benchmark for evaluating the \nGovernment’s conduct of financial policy.  The statement is also consistent with section \n11(1)(a) of the Act. \nStrategic Priorities and Financial Policy \nIn this budget, the Government continues its commitment to the principles of responsible \nfinancial management. \nThe financial objectives and the key measures below outline the Government’s financial \npolicy.  Strategic priorities, as they relate to the Territory’s Budget, are summarised as: \n• maintain a balanced budget over the economic cycle; \n• maintain low levels of debt; \n• provide the highest possib

In [83]:
# Get specific text by Keywords
keywords = ["budget", "debt", "infrastructure"]

import re

def extract_sentences_with_keywords(pages, keywords):
    results = []
    for page in pages:
        sentences = re.split(r'(?<=[.!?]) +', page["content"])
        for sent in sentences:
            lower_sent = sent.lower()
            for kw in keywords:
                if kw in lower_sent:
                    results.append({
                        "keyword": kw,
                        "page": page["page"],
                        "excerpt": sent
                    })
    return results

important_info = extract_sentences_with_keywords(pages, keywords)
for info in important_info:
    print(info)


{'keyword': 'budget', 'page': 1, 'excerpt': '1.2 \nFINANCIAL POLICY OBJECTIVES AND STRATEGIES \n \nSTATEMENT \nThe presentation and preparation of the Territory’s Budget is provided for in sections 11 and \n11A of the Financial Management Act 1996 (the Act).'}
{'keyword': 'budget', 'page': 1, 'excerpt': '\nStrategic Priorities and Financial Policy \nIn this budget, the Government continues its commitment to the principles of responsible \nfinancial management.'}
{'keyword': 'budget', 'page': 1, 'excerpt': 'Strategic priorities, as they relate to the Territory’s Budget, are summarised as: \n• maintain a balanced budget over the economic cycle; \n• maintain low levels of debt; \n• provide the highest possible standard of government services; \n• service delivery which focuses on people, the environment and building prosperity;  \n• maintain a triple A credit rating; and \n• effective integration of economic and environmental considerations to promote \nsustainability of service delivery.

In [84]:
!pip install transformers sentence-transformers faiss-cpu accelerate
!pip install faiss-cpu sentence-transformers
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [85]:
# Convert text to vector
data = important_info

model = SentenceTransformer("all-MiniLM-L6-v2")


texts = [item["excerpt"] for item in data]
embeddings = model.encode(texts, convert_to_numpy=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"Stored {index.ntotal} policy chunks in FAISS")

Stored 39 policy chunks in FAISS


In [86]:
# Answer the Question
qa_model = pipeline("text2text-generation", model="google/flan-t5-base")

conversation_history = []

def search(query, top_k=5):
    query_vec = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    results = [data[i] for i in indices[0]]
    return results

def chatbot_response(user_input):
    conversation_history.append({"role": "user", "content": user_input})

    results = search(user_input)
    context_text = "\n".join([f"Page {r['page']}: {r['excerpt']}" for r in results])

    prompt = f"""
You are a helpful assistant. Use only the information below.
Provide a 2–3 sentence answer.

Excerpts:
{context_text}

Question: {user_input}
Answer:
"""

    reply = qa_model(prompt, max_new_tokens=256, do_sample=False)[0]["generated_text"]

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

Device set to use cpu


In [87]:
# Sample example
print(chatbot_response("What is the budget?"))
print(chatbot_response("And what about debt?"))
print(chatbot_response("What about infrastructure?"))

2005-06 Budget Paper No.
The objectives of a balanced budget over the economic cycle, the maintenance of low levels of debt and adequate provision for long-term liabilities strongly support prudent financial management and the principles of inter-generational equity.
A strategic approach to capital works programs reflects a longer term view of key infrastructure decisions such as the timely delivery of new infrastructure and the timely replacement of infrastructure balanced against the changing demographics and changing service delivery needs.
